In [ ]:
import random

from rdkit import Chem

from stereomolgraph import AtomId, Bond, MolGraph, StereoMolGraph
from stereomolgraph.experimental.canon import canon_atom_num
from stereomolgraph.ipython import View2D

View2D.show_h = True

In [ ]:
def shuffle_graph(g: StereoMolGraph | MolGraph) -> StereoMolGraph | MolGraph:
    random_atom_ids: list[AtomId] = random.sample(range(len(g)), len(g))

    old_new_mapping: dict[AtomId | None, AtomId | None] = {None: None}
    new_old_mapping: dict[AtomId | None, AtomId | None] = {None: None}

    new_g = g.__class__()

    for atom, atom_type in random.sample(list(zip(g.atoms, g.atom_types)), len(g)):
        new_atom_id = random_atom_ids.pop()
        old_new_mapping[atom] = new_atom_id
        new_old_mapping[new_atom_id] = atom
        new_g.add_atom(new_atom_id, atom_type)

    for a1, a2 in random.sample(list(g.bonds), len(g.bonds)):
        new_g.add_bond(old_new_mapping[a1], old_new_mapping[a2])

    if isinstance(g, StereoMolGraph):
        for stereo in g.atom_stereo.values():
            new_stereo = stereo.__class__(
                tuple(old_new_mapping.get(atom) for atom in stereo.atoms), stereo.parity
            )

            new_g.set_atom_stereo(new_stereo)

        for stereo in g.bond_stereo.values():
            new_stereo = stereo.__class__(
                tuple(old_new_mapping.get(atom) for atom in stereo.atoms), stereo.parity
            )

            new_g.set_bond_stereo(new_stereo)
    assert new_g == g  # This is computationally moderately expensive
    return new_g


In [ ]:
from stereomolgraph.stereodescriptors import OInt


def graph_to_set(
    g: StereoMolGraph | MolGraph,
) -> set[tuple[AtomId, int] | Bond | tuple[str, int, tuple[OInt, ...]]]:
    ret: set[tuple[AtomId, int] | Bond | tuple[str, int, tuple[OInt, ...]]] = {
        (a, t) for a, t in zip(g.atoms, g.atom_types)
    }
    for bond in g.bonds:
        ret.add(bond)
    if isinstance(g, StereoMolGraph):
        for stereo in g.stereo.values():
            if None in stereo.atoms:
                # chiral lone pairs are described using "None"
                # To make it comparible it is substituted by len(graph)
                stereo = stereo.__class__(
                    tuple(a if a is not None else len(g) + 1 for a in stereo.atoms),
                    stereo.parity,
                )
            ret.add((stereo.__class__.__name__, *stereo.canonical_form()))

    if isinstance(g, StereoMolGraph):
        assert len(ret) == len(g.atoms) + len(g.bonds) + len(g.stereo)

    elif isinstance(g, MolGraph):
        assert len(ret) == len(g.atoms) + len(g.bonds)

    return ret

In [ ]:
def smiles2graph(smiles: str) -> MolGraph:
    rdmol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    # mg = MolGraph.from_rdmol(rdmol)  # without stereochenmistry
    # currently a little buggy to color refinement
    mg = StereoMolGraph.from_rdmol(rdmol, stereo_complete=True)
    return mg

In [ ]:
def test_canon_atom_num(g: MolGraph, n: int = 10) -> bool:
    distinct_graphs = []
    relabeled_graphs: list[MolGraph] = []
    for _ in range(n):
        shuffeled_g = shuffle_graph(g)
        canon_id_mapping = canon_atom_num(shuffeled_g)
        canon_graph = shuffeled_g.relabel_atoms(canon_id_mapping)
        graph_as_set = graph_to_set(canon_graph)
        if graph_as_set not in distinct_graphs:
            distinct_graphs.append(graph_as_set)
            relabeled_graphs.append(canon_graph)

    if len(distinct_graphs) > 1:
        display(relabeled_graphs[0])
        display(relabeled_graphs[1])
        print(distinct_graphs[0].symmetric_difference(distinct_graphs[1]))
    return True if len(distinct_graphs) == 1 else False

Difficult graphs from: 
Krotko, D.G. Atomic ring invariant and Modified CANON extended connectivity algorithm for symmetry perception in molecular graphs and rigorous canonicalization of SMILES. J Cheminform 12, 48 (2020). https://doi.org/10.1186/s13321-020-00453-4

In [ ]:
difficult_graphs = [
    "C1OC23COC45COC11COC67COC8(COC9(CO2)COC(CO1)(CO6)OCC(CO9)(OC4)OCC(CO5)(OC7)OC8)OC3",
    "C12C3C4C5C1C6C7C2C8C3C6C5C8C74",  # peterson graph G(7,2)
    "C1(C2C3C4C15)C6C7C2C8C3C9C%10C4C%11C5C6C%12C%11C%10C%13C%12C7C8C9%13",
    "C1CC2CCC1CCC3CCC(CC3)CC2",
    "C12C3C1C4C5C4C5C23",
    "C1C2CC3CC1CC(C2)C3",  # adamantane
    "C12C3C4C1C5C2C3C45",  # cubane
]

smiles_list from: https://cipvalidationsuite.github.io/ValidationSuite/

In [ ]:
# Read SMILES from text file
def load_smiles_from_txt(filepath: str) -> list[str]:
    """Load SMILES strings from a text file"""
    with open(filepath, "r") as f:
        smiles_list = [line.strip() for line in f if line.strip()]
    print(f"Loaded {len(smiles_list)} SMILES from {filepath}")
    return smiles_list


ids_to_skip = [
    7,
    22,
    119,
    175,
    176,
    177,
    178,
    179,
    180,
    181,
    182,
    183,
    184,
    185,
    186,
    187,  # Isotopes
    10,
    11,  # Helicenes
    63,
    78,
    79,
    118,
    120,
    135,
    141,
    144,
    154,
    164,
    166,
    231,
    232,
    243,
    287,  # Allenes, Commulenes
    23,
    55,
    57,
    72,
    73,
    86,
    124,
    155,
    158,  # Atrop
]

# Load SMILES from file
smiles_file = load_smiles_from_txt("./smiles.txt")
smiles_list = [
    smiles_file[i - 1].split()[0]
    for i, _ in enumerate(smiles_file, start=1)
    if i not in ids_to_skip
]

In [ ]:
for smiles in difficult_graphs + smiles_list:
    try:
        g = smiles2graph(smiles)
    except Exception as e:
        print(f"Error processing SMILES: {smiles}: {e}")
        continue

    print(smiles, end="\r", flush=True)
    if test_canon_atom_num(g):
        print(f"PASS: {smiles}")
    else:
        print(f"FAIL: {smiles}")
